<a href="https://colab.research.google.com/github/Nelcy2005/footwear-accessories-genai/blob/main/MDS510A5_Footwear_GenAI_2548419.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Personal Parameters & Setup**

In [3]:
!pip install -q datasets transformers scikit-learn pandas

import random, re
import numpy as np
import pandas as pd
import tensorflow as tf

# Personal Parameters (Register No: 2548419)
SEED = 8419
TOP_K = 50
GPT2_LR = 3e-5
IMG_SIZE = 64
LATENT_DIM = 64
VAE_BETA = 1.0
DIFFUSION_T = 500
NOISE_SCHEDULE = "linear"
IMG_SAMPLE_SIZE = 1100

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Parameters and seeds locked:", SEED)

Parameters and seeds locked: 8419


**Cell 2: Build Corpus (≥800 Dataset Sentences + Your 50) & 80/10/10 Split**

In [4]:
import urllib.request
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# 1. Load metadata from Fashion Product Images (Small)
ds = load_dataset("ashraq/fashion-product-images-small", split="train")
df = ds.remove_columns(["image"]).to_pandas()

# Filter to Footwear & Accessories and build clean sentences
df = df[df["masterCategory"].isin(["Footwear", "Accessories"])].dropna(subset=["productDisplayName"])

def make_sentence(row):
    text = f"{row['productDisplayName']} is a {row['baseColour']} {row['articleType']} for {row['gender']} ({row['usage']} wear)."
    return re.sub(r"\s+", " ", text).strip()

dataset_sentences = df.sample(n=1000, random_state=SEED).apply(make_sentence, axis=1).tolist()

# 2. Load your 50+ hand-written sentences from GitHub
raw_url = "https://raw.githubusercontent.com/Nelcy2005/footwear-accessories-genai/main/data/my_50_sentences.txt"
my_sentences = [line.decode("utf-8").strip() for line in urllib.request.urlopen(raw_url) if line.strip()]

# 3. Combine & 80/10/10 split
corpus = list(dict.fromkeys(dataset_sentences + my_sentences))  # remove exact duplicates
train_txt, temp_txt = train_test_split(corpus, test_size=0.2, random_state=SEED)
val_txt, test_txt = train_test_split(temp_txt, test_size=0.5, random_state=SEED)

# Interactive summary table
pd.DataFrame({
    "Split": ["Dataset sentences", "My written sentences", "Total unique", "Train (80%)", "Val (10%)", "Test (10%)"],
    "Count": [len(dataset_sentences), len(my_sentences), len(corpus), len(train_txt), len(val_txt), len(test_txt)]
})

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…): reconstructing file:   0%|          |  0.00B /  136MB            

data/train-00000-of-00002-6cff4c59f91661(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-bb459e5ac5f01e(…): reconstructing file:   0%|          |  0.00B /  135MB            

data/train-00001-of-00002-bb459e5ac5f01e(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

,Split,Count
0,Dataset sentences,1000
1,My written sentences,5
2,Total unique,957
3,Train (80%),765
4,Val (10%),96
5,Test (10%),96


**Cell 3: BPE (GPT-2) vs WordPiece (BERT) Tokenisation Comparison**

In [5]:
from transformers import AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# 1. Compare average tokens per sentence across corpus
gpt2_lens = [len(gpt2_tok.tokenize(s)) for s in corpus]
bert_lens = [len(bert_tok.tokenize(s)) for s in corpus]

summary = pd.DataFrame({
    "Tokeniser": ["GPT-2 (BPE)", "BERT (WordPiece)"],
    "Avg Tokens / Sentence": [round(np.mean(gpt2_lens), 2), round(np.mean(bert_lens), 2)],
    "Max Tokens": [max(gpt2_lens), max(bert_lens)],
    "Min Tokens": [min(gpt2_lens), min(bert_lens)]
})
display(summary)

# 2. Compare how domain terms split into sub-words
domain_terms = ["loafers", "sneakers", "chronograph", "crossbody", "stiletto", "espadrilles", "waterproof"]
splits = pd.DataFrame({
    "Domain Term": domain_terms,
    "GPT-2 (BPE)": [gpt2_tok.tokenize(w) for w in domain_terms],
    "BERT (WordPiece)": [bert_tok.tokenize(w) for w in domain_terms]
})
display(splits)

,Tokeniser,Avg Tokens / Sentence,Max Tokens,Min Tokens
0,GPT-2 (BPE),21.14,38,11
1,BERT (WordPiece),19.99,37,11


,Domain Term,GPT-2 (BPE),BERT (WordPiece)
0,loafers,"[lo, af, ers]","[loaf, ##ers]"
1,sneakers,"[s, ne, akers]",[sneakers]
2,chronograph,"[chron, ograph]","[ch, ##ron, ##og, ##raph]"
3,crossbody,"[cross, body]","[cross, ##body]"
4,stiletto,"[st, ile, tto]","[stil, ##etto]"
5,espadrilles,"[esp, ad, rill, es]","[es, ##pad, ##rill, ##es]"
6,waterproof,"[water, proof]","[water, ##proof]"
